In [1]:
# ==========================================
# STEP 1: ALL REQUIRED IMPORTS
# ==========================================
import os
import zipfile
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from google.colab import userdata
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.layers import Input, Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras import mixed_precision

print("TensorFlow Version:", tf.__version__)

# Set Seed for Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ==========================================
# STEP 2: KAGGLE DATASET DOWNLOAD & EXTRACTION
# ==========================================
# Kaggle secrets check (Make sure KAGGLE_USERNAME & KAGGLE_KEY are in Colab Secrets)
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

if os.environ["KAGGLE_USERNAME"] is None or os.environ["KAGGLE_KEY"] is None:
    raise ValueError("Please add KAGGLE_USERNAME and KAGGLE_KEY in Colab Secrets (Left Key Icon).")

print("Kaggle secrets loaded. Downloading dataset...")
!kaggle datasets download -d jangedoo/utkface-new -p /content --force

print("Extracting dataset...")
zip_path = "/content/utkface-new.zip"
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall("/content")

DATASET_DIR = "/content/utkface_aligned_cropped/UTKFace"
print("Total images found:", len(os.listdir(DATASET_DIR)))

# ==========================================
# STEP 3: CREATE DATAFRAME & SPLIT
# ==========================================
data = []
for filename in os.listdir(DATASET_DIR):
    try:
        parts = filename.split("_")
        age = int(parts[0])
        gender = int(parts[1])
        filepath = os.path.join(DATASET_DIR, filename)
        data.append([filepath, age, gender])
    except Exception:
        continue

df = pd.DataFrame(data, columns=["filepath", "age", "gender"])
print("DataFrame Shape:", df.shape)

# Train (80%), Val (10%), Test (10%) Split
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=SEED)

# ==========================================
# STEP 4: OPTIMIZED SPEED PIPELINE (tf.data)
# ==========================================
# 10x Fast Epochs ke liye Mixed Precision Enable karein
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)

# ResNet50 input size standard 224x224 hota hai
IMG_SIZE = (224, 224)

def process_data(filepath, age, gender):
    img = tf.io.read_file(filepath)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = preprocess_input(img)  # ResNet50 standard preprocessing
    return img, {'age_output': tf.cast(age, tf.float32), 'gender_output': tf.cast(gender, tf.float32)}

def create_dataset(dataframe, batch_size=64, is_training=True):
    dataset = tf.data.Dataset.from_tensor_slices((
        dataframe['filepath'].values,
        dataframe['age'].values,
        dataframe['gender'].values
    ))
    AUTOTUNE = tf.data.AUTOTUNE
    dataset = dataset.map(process_data, num_parallel_calls=AUTOTUNE)
    if is_training:
        dataset = dataset.shuffle(buffer_size=1000).repeat()
    dataset = dataset.batch(batch_size).prefetch(buffer_size=AUTOTUNE)
    return dataset

BATCH_SIZE = 64
train_ds = create_dataset(train_df, batch_size=BATCH_SIZE, is_training=True)
val_ds = create_dataset(val_df, batch_size=BATCH_SIZE, is_training=False)
test_ds = create_dataset(test_df, batch_size=BATCH_SIZE, is_training=False)

# ==========================================
# STEP 5: RESNET50 MULTI-OUTPUT MODEL ARCHITECTURE
# ==========================================
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False  # Freeze pre-trained weights for speed

inputs = Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.4)(x)

# Outputs definition (Note: Mixed precision ke liye dtype='float32' dena zaroori hai)
age_output = Dense(1, activation='linear', dtype='float32', name='age_output')(x)
gender_output = Dense(1, activation='sigmoid', dtype='float32', name='gender_output')(x)

model = Model(inputs=inputs, outputs=[age_output, gender_output])

# Compile model with separate losses
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss={'age_output': 'mse', 'gender_output': 'binary_crossentropy'},
    metrics={'age_output': 'mae', 'gender_output': 'accuracy'}
)

# ==========================================
# STEP 6: MODEL TRAINING (model.fit)
# ==========================================
callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    ModelCheckpoint('best_resnet_age_gender.h5', monitor='val_loss', save_best_only=True)
]

steps_per_epoch = len(train_df) // BATCH_SIZE

print("\n--- TRAINING SHURU HO RAHI HAI (ResNet50 GPU Speed Check) ---")
history = model.fit(
    train_ds,
    steps_per_epoch=steps_per_epoch,
    epochs=15,
    validation_data=val_ds,
    callbacks=callbacks
)

# ==========================================
# STEP 7: EVALUATE ON TEST SET
# ==========================================
print("\n--- MODEL EVALUATION ON UNSEEN TEST DATA ---")
results = model.evaluate(test_ds)
print(f"Age Average Error (MAE): {results[3]:.2f} Years")
print(f"Gender Prediction Accuracy: {results[4]*100:.2f}%")


TensorFlow Version: 2.20.0
Kaggle secrets loaded. Downloading dataset...
Dataset URL: https://www.kaggle.com/datasets/jangedoo/utkface-new
License(s): copyright-authors
100% 331M/331M [00:04<00:00, 83.1MB/s]

Extracting dataset...
Total images found: 23708
DataFrame Shape: (23708, 3)
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step

--- TRAINING SHURU HO RAHI HAI (ResNet50 GPU Speed Check) ---
Epoch 1/15
296/296 ━━━━━━━━━━━━━━━━━━━━ 0s 147ms/step - age_output_loss: 389.3501 - age_output_mae: 14.1635 - gender_output_accuracy: 0.5779 - gender_output_loss: 1.0542 - loss: 390.4043

296/296 ━━━━━━━━━━━━━━━━━━━━ 81s 201ms/step - age_output_loss: 213.7458 - age_output_mae: 10.5759 - gender_output_accuracy: 0.6405 - gender_output_loss: 0.8903 - loss: 214.6361 - val_age_output_loss: 112.1226 - val_age_output_mae: 7.9539 - val_gender_output_accuracy: 0.8051 - val_gender_output_loss: 0.4356 - val_loss: 114.3393
Epoch 2/15
296/296 ━━━━━━━━━━━━━━━━━━━━ 0s 150ms/step - age_output_loss: 124.1090 - age_output_mae: 8.3092 - gender_output_accuracy: 0.7257 - gender_output_loss: 0.5910 - loss: 124.7000

296/296 ━━━━━━━━━━━━━━━━━━━━ 49s 167ms/step - age_output_loss: 122.5460 - age_output_mae: 8.2208 - gender_output_accuracy: 0.7286 - gender_output_loss: 0.5677 - loss: 123.1136 - val_age_output_loss: 100.0342 - val_age_output_mae: 7.4744 - val_gender_output_accuracy: 0.8136 - val_gender_output_loss: 0.4353 - val_loss: 102.1775
Epoch 3/15
296/296 ━━━━━━━━━━━━━━━━━━━━ 0s 145ms/step - age_output_loss: 113.2420 - age_output_mae: 7.9333 - gender_output_accuracy: 0.7447 - gender_output_loss: 0.5164 - loss: 113.7584

296/296 ━━━━━━━━━━━━━━━━━━━━ 54s 182ms/step - age_output_loss: 111.8553 - age_output_mae: 7.8310 - gender_output_accuracy: 0.7494 - gender_output_loss: 0.5143 - loss: 112.3696 - val_age_output_loss: 95.9361 - val_age_output_mae: 7.2584 - val_gender_output_accuracy: 0.8313 - val_gender_output_loss: 0.4006 - val_loss: 98.1742
Epoch 4/15
296/296 ━━━━━━━━━━━━━━━━━━━━ 49s 167ms/step - age_output_loss: 107.6003 - age_output_mae: 7.6724 - gender_output_accuracy: 0.7600 - gender_output_loss: 0.4936 - loss: 108.0938 - val_age_output_loss: 96.9631 - val_age_output_mae: 7.1663 - val_gender_output_accuracy: 0.8161 - val_gender_output_loss: 0.4107 - val_loss: 98.8667
Epoch 5/15
296/296 ━━━━━━━━━━━━━━━━━━━━ 50s 170ms/step - age_output_loss: 103.5856 - age_output_mae: 7.5227 - gender_output_accuracy: 0.7655 - gender_output_loss: 0.4771 - loss: 104.0626 - val_age_output_loss: 111.4299 - val_age_output_mae: 7.7362 - val_gender_output_accuracy: 0.8283 - val_gender_output_loss: 0.3987 - val_loss: 113.327

296/296 ━━━━━━━━━━━━━━━━━━━━ 50s 168ms/step - age_output_loss: 101.7381 - age_output_mae: 7.4546 - gender_output_accuracy: 0.7687 - gender_output_loss: 0.4745 - loss: 102.2126 - val_age_output_loss: 91.5736 - val_age_output_mae: 7.0686 - val_gender_output_accuracy: 0.8148 - val_gender_output_loss: 0.4128 - val_loss: 93.9061
Epoch 7/15
296/296 ━━━━━━━━━━━━━━━━━━━━ 49s 165ms/step - age_output_loss: 96.4380 - age_output_mae: 7.2669 - gender_output_accuracy: 0.7740 - gender_output_loss: 0.4697 - loss: 96.9077 - val_age_output_loss: 93.1916 - val_age_output_mae: 7.2735 - val_gender_output_accuracy: 0.8342 - val_gender_output_loss: 0.3788 - val_loss: 95.8149
Epoch 8/15
296/296 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step - age_output_loss: 96.4732 - age_output_mae: 7.2967 - gender_output_accuracy: 0.7888 - gender_output_loss: 0.4480 - loss: 96.9212

296/296 ━━━━━━━━━━━━━━━━━━━━ 48s 163ms/step - age_output_loss: 96.5697 - age_output_mae: 7.2597 - gender_output_accuracy: 0.7809 - gender_output_loss: 0.4577 - loss: 97.0274 - val_age_output_loss: 87.4691 - val_age_output_mae: 6.9227 - val_gender_output_accuracy: 0.8262 - val_gender_output_loss: 0.3778 - val_loss: 89.8160
Epoch 9/15
296/296 ━━━━━━━━━━━━━━━━━━━━ 48s 162ms/step - age_output_loss: 95.3400 - age_output_mae: 7.2408 - gender_output_accuracy: 0.7820 - gender_output_loss: 0.4536 - loss: 95.7936 - val_age_output_loss: 89.0902 - val_age_output_mae: 7.0067 - val_gender_output_accuracy: 0.8397 - val_gender_output_loss: 0.3600 - val_loss: 91.4337
Epoch 10/15
296/296 ━━━━━━━━━━━━━━━━━━━━ 46s 156ms/step - age_output_loss: 92.0804 - age_output_mae: 7.0865 - gender_output_accuracy: 0.7801 - gender_output_loss: 0.4545 - loss: 92.5349 - val_age_output_loss: 94.0309 - val_age_output_mae: 7.1142 - val_gender_output_accuracy: 0.8258 - val_gender_output_loss: 0.3916 - val_loss: 96.1744
Epoch

In [ ]:
import gradio as gr
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications.resnet50 import preprocess_input

# 1. Save kiya hua ResNet50 model load karein
# Note: Agar aap ne model ka naam badla tha, to yahan sahi file name '.h5' dein
print("Trained ResNet50 Model load ho raha hai...")
model = tf.keras.models.load_model('best_resnet_age_gender.h5', compile=False)

# 2. Prediction karne ka function define karein
def predict_age_and_gender(input_image):
    if input_image is None:
        return "Image nahi mili", "Image nahi mili"

    # Gradio image ko numpy array mein deta hai, isay model ke size (224x224) par resize karein
    img = tf.image.resize(input_image, (224, 224))
    img_array = tf.expand_dims(img, axis=0) # Batch dimension add karein (1, 224, 224, 3)

    # ResNet50 standard preprocessing apply karein
    img_preprocessed = preprocess_input(img_array)

    # Model se output (predictions) lein
    age_pred, gender_pred = model.predict(img_preprocessed)

    # A) Age calculation (Regression values ko round off kar ke integer banayein)
    estimated_age = int(np.round(age_pred[0][0]))
    if estimated_age < 0: estimated_age = 0 # Age negative na ho

    # B) Gender calculation (Sigmoid output -> 0 to 1)
    # 0 = Male, 1 = Female
    gender_prob = gender_pred[0][0]
    if gender_prob >= 0.5:
        gender_result = f"Female ({gender_prob * 100:.1f}%)"
    else:
        gender_result = f"Male ({(1 - gender_prob) * 100:.1f}%)"

    return f"🎂 {estimated_age} Years", f"👤 {gender_result}"

# 3. Gradio Interface Layout Elements Setup
interface = gr.Interface(
    fn=predict_age_and_gender,
    inputs=gr.Image(label="Apni Face Image Upload Karein"),
    outputs=[
        gr.Textbox(label="Estimated Age (Umar)"),
        gr.Textbox(label="Predicted Gender (Jins)")
    ],
    title="🧠 AI Age & Gender Predictor (ResNet50)",
    description="Is Web App mein apni ya kisi ki bhi face image upload karein. Humara trained Multi-Output ResNet50 Deep Learning model btaega ke tasveer mein mojood shakhs ki umar kya hai aur woh Male hai ya Female.",
    theme="soft"
)

# 4. Web App ko launch karein
# share=True lagane se aap ko aik public URL mil jaye ga jise aap doston ke sath share kar sakte hain
interface.launch(share=True, debug=True)


Trained ResNet50 Model load ho raha hai...


/usr/local/lib/python3.13/dist-packages/gradio/interface.py:171: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  super().__init__(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d9fcec2376cd3ea55e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


1/1 ━━━━━━━━━━━━━━━━━━━━ 9s 9s/step
